# Pythia 2.8b: Accessibility Domain Knowledge

32 layers | 32 heads | d_model=2560 | d_mlp=10240 | parallel attn+MLP

In [27]:
import sys
sys.path.insert(0, '../../..')

from src.models import load_model, unload
from src.logit_lens import logit_lens, print_logit_lens
from src.decompose import decompose_contributions, print_decomposition, summarize_contributions
from src.heads import per_head_contributions, top_heads, print_top_heads, print_layer_summary
from src.viz import plot_logit_lens, plot_decomposition, plot_head_heatmap, save_figures

In [28]:
model_name = "pythia-2.8b"
model, info = load_model(model_name)

Loaded pretrained model pythia-2.8b into HookedTransformer
Loaded pythia-2.8b on mps
  32 layers | 32 heads | d_model=2560 | d_mlp=10240 | parallel attn+MLP


In [29]:
prompts = [
    ("The W in WCAG stands for", " Web", "wcag"),
    ("ARIA stands for Accessible Rich Internet", " Applications", "aria"),
    ("The purpose of alt text is to describe an", " image", "alt"),
    ("In HTML, the alt attribute provides a text description of an", " image", "HTML"),
    ("A blind person uses a screen", " reader", "screenReader"),
    ("A screen reader is", " software", "screenReaderv1"),
]

results = {}
for prompt, target, label in prompts:
    print(f"\n{'='*60}")
    print(f"  {label.upper()}: \"{prompt}\" → \"{target}\"")
    print(f"{'='*60}\n")

    df_lens, cache = logit_lens(model, prompt, target)
    print_logit_lens(df_lens)

    df_decomp = decompose_contributions(model, cache, target)
    print_decomposition(df_decomp)
    print(summarize_contributions(df_decomp))

    df_heads = per_head_contributions(model, cache, target)
    print_top_heads(df_heads)

    results[label] = {"lens": df_lens, "decomp": df_decomp, "heads": df_heads}


  WCAG: "The W in WCAG stands for" → " Web"

Model: pythia-2.8b
Prompt: "The W in WCAG stands for"
Target: " Web" (token 7066)
Final prediction: " Web"

Layer    Top Prediction       Target Rank    Target Prob 
------------------------------------------------------
0        ibase                20931          0.000000    
1        pril                 29021          0.000000    
2        pril                 26514          0.000000    
3        pril                 33983          0.000000    
4        =”                   37274          0.000000    
5        IRST                 41552          0.000000    
6        IRST                 42959          0.000000    
7        IRST                 45855          0.000000    
8        IRST                 45308          0.000000    
9        IRST                 47283          0.000000    
10       IRST                 43399          0.000000    
11       HECK                 35896          0.000001    
12       IRST                 42295  

In [30]:
for label, data in results.items():
    plt.close(plot_logit_lens(data["lens"]))
    plt.close(plot_decomposition(data["decomp"]))
    plt.close(plot_head_heatmap(data["heads"]))

In [31]:
for label, data in results.items():
    data["lens"].to_csv(f"../../results/pythia/{model_name}/{label}-logit-lens.csv", index=False)
    data["decomp"].to_csv(f"../../results/pythia/{model_name}/{label}-decomposition.csv", index=False)
    data["heads"].to_csv(f"../../results/pythia/{model_name}/{label}-heads.csv", index=False)
    save_figures(model_name, label,
                 logit_lens=data["lens"], decomposition=data["decomp"], heads=data["heads"],
                 out_dir=f"../../results/pythia/{model_name}/figures")

Saved: ../../results/pythia/pythia-2.8b/figures/wcag-logit-lens.png
Saved: ../../results/pythia/pythia-2.8b/figures/wcag-decomposition.png
Saved: ../../results/pythia/pythia-2.8b/figures/wcag-head-heatmap.png
Saved: ../../results/pythia/pythia-2.8b/figures/aria-logit-lens.png
Saved: ../../results/pythia/pythia-2.8b/figures/aria-decomposition.png
Saved: ../../results/pythia/pythia-2.8b/figures/aria-head-heatmap.png
Saved: ../../results/pythia/pythia-2.8b/figures/alt-logit-lens.png
Saved: ../../results/pythia/pythia-2.8b/figures/alt-decomposition.png
Saved: ../../results/pythia/pythia-2.8b/figures/alt-head-heatmap.png
Saved: ../../results/pythia/pythia-2.8b/figures/HTML-logit-lens.png
Saved: ../../results/pythia/pythia-2.8b/figures/HTML-decomposition.png
Saved: ../../results/pythia/pythia-2.8b/figures/HTML-head-heatmap.png
Saved: ../../results/pythia/pythia-2.8b/figures/screenReader-logit-lens.png
Saved: ../../results/pythia/pythia-2.8b/figures/screenReader-decomposition.png
Saved: ../..

In [33]:
unload(model)

Model unloaded, memory cleared
